# Reddit Market Research Agent with Scavio API

Mine Reddit recommendation threads to discover what products and tools people actually use and recommend. Uses the Scavio search API and LangChain to extract common picks, pain points, and unmet needs from real discussions. A free alternative to GummySearch and SparkToro.

**What you will learn:**
- Search Reddit for recommendation threads with ScavioRedditSearch
- Read full threads and comments with ScavioRedditPost
- Extract product recommendations and pain points
- Build a market research report from real user opinions

**Prerequisites:**
- Free Scavio API key (250 credits/month): https://dashboard.scavio.dev
- OpenAI API key

**Tools used:** ScavioRedditSearch, ScavioRedditPost

In [1]:
# pip install langchain langchain-openai langchain-scavio python-dotenv

In [2]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_scavio import ScavioRedditSearch, ScavioRedditPost

load_dotenv(override=True)

True

In [3]:
SYSTEM_PROMPT = """You are RedditMarketResearch, a product and market research agent.

Workflow:
1. Take the user's product category or market niche.
2. Call ScavioRedditSearch for "<niche> recommendation" to find
   recommendation threads. Sort by "new".
3. Call ScavioRedditSearch for "best <niche>" to find comparison threads.
4. Deduplicate by post ID. Pick the top 3-5 most relevant threads.
5. Call ScavioRedditPost on the top 3 threads to read full comments.
6. Extract insights and produce:

   ## Reddit Market Research: <niche>

   ### Most Recommended Products/Tools
   Ranked list of products mentioned across threads:
   - **<Product>** -- mentioned X times, common praise: <summary>

   ### Pain Points
   What problems do users complain about in this category?
   - <pain point 1>
   - <pain point 2>

   ### Unmet Needs
   What are users asking for that no current product solves?
   - <unmet need>

   ### Key Threads
   Top 3 threads with title, subreddit, URL, and key takeaway.

   ### Market Opportunity
   One paragraph: what product or feature would win this market
   based on Reddit feedback.

Rules:
- Never invent thread titles, URLs, subreddits, or product names.
- Call only ONE tool per step.
- Keep the report under 400 words.
"""

In [4]:
def build_agent():
    model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    tools = [
        ScavioRedditSearch(max_results=10),
        ScavioRedditPost(),
    ]
    return create_agent(model, tools=tools, system_prompt=SYSTEM_PROMPT)

In [5]:
agent = build_agent()
result = agent.invoke({
    "messages": [{"role": "user", "content": "project management tools for small teams"}]
})
print(result["messages"][-1].content)

## Reddit Market Research: Project Management Tools for Small Teams

### Most Recommended Products/Tools
- **Chaser** -- mentioned multiple times, praised for seamless Slack integration, ease of use, and effective day-to-day task tracking inside Slack conversations.
- **Trello** -- highly recommended for simplicity, visual task management, and good Slack integration; great for adoption and lightweight workflows.
- **Asana** -- valued for structured project execution with dependencies, milestones, and approvals; moderate Slack integration.
- **Monday.com** -- liked for visual spreadsheet-like interface and PM fundamentals; Slack integration supports notifications and task creation.
- **Celoxis** -- recommended for all-in-one Gantt, resource tracking, and reporting; good for teams needing more than task tracking.
- **Wrike** -- deep feature set including Gantt, dependencies, and resource management; steeper learning curve.
- **Airtable** -- flexible database-style PM, powerful but requir

## Next Steps

- Research any product category or market niche on Reddit
- Compare recommendations across different subreddit communities
- Track how product sentiment changes over time
- Use findings to validate startup ideas or product features

**Credits used:** ~4-6 per run (two searches + three post reads)